# Credit Risk Assessment & Loan Default Prediction

#### Bussiness Problem: Financial institutions need to assess borrower risk before approving loans. This project predicts the likelihood of loan default using machine learning models.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option('display.float_format', lambda x: '{:.2f}'.format(x))
np.set_printoptions(suppress=True)

# DATA GATHERING:

In [ ]:
# Load datasets
df_customers=pd.read_csv("customers.csv")
df_loans=pd.read_csv("loans.csv")
df_bureau=pd.read_csv("bureau_data.csv")

In [ ]:
df_customers.shape,df_loans.shape,df_bureau.shape

In [ ]:
df_customers.info()

In [ ]:
df_customers.head(3)

In [ ]:
df_loans.info()

In [ ]:
df_loans.head(3)

In [ ]:
df_bureau.info()

In [ ]:
df_bureau.head(3)

In [ ]:
df=pd.merge(df_customers,df_loans,on='cust_id')
df.head(3)

In [ ]:
df.info()

In [ ]:
df=pd.merge(df,df_bureau,on='cust_id')

In [ ]:
df.info()

In [ ]:
df['default']=df['default'].astype(int)
df.default.value_counts()

In [ ]:
X=df.drop("default",axis="columns")
y=df['default']

X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y,test_size=0.25,random_state=42)

df_train=pd.concat([X_train,y_train],axis="columns")
df_test=pd.concat([X_test,y_test],axis="columns")

In [ ]:
df_train.info()

# DATA PREPROCESSING:

In [ ]:
df_train.shape

In [ ]:
df_train.isna().sum()

In [ ]:
df_train.residence_type.unique()

In [ ]:
mode_residence=df_train.residence_type.mode()[0]
mode_residence

In [ ]:
df_train["residence_type"]=df_train["residence_type"].fillna(mode_residence)
df_test["residence_type"]=df_test["residence_type"].fillna(mode_residence)

In [ ]:
df_train.isna().sum()

In [ ]:
df_train.residence_type.unique()

In [ ]:
df_train.duplicated().sum()

In [ ]:
df_train.columns

In [ ]:
df_train.dtypes

In [ ]:
df_train.shape

In [ ]:
columns_continuous = ['age','income','number_of_dependants','years_at_current_address', 
                      'sanction_amount','loan_amount','processing_fee','gst','net_disbursement', 
                      'loan_tenure_months','principal_outstanding', 'bank_balance_at_application',
                      'number_of_open_accounts','number_of_closed_accounts','total_loan_months','delinquent_months',
                       'total_dpd','enquiry_count','credit_utilization_ratio']

columns_categorical = ['gender','marital_status','employment_status','residence_type','city', 
                       'state','zipcode','loan_purpose','loan_type','default']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

num_plots=len(columns_continuous)
num_cols=4   # number of plots per row
num_rows=math.ceil(num_plots/num_cols)  # ceiling division for rows

plt.figure(figsize=(20,5*num_rows))
for i,col in enumerate(columns_continuous,1):
    plt.subplot(num_rows,num_cols,i)
    sns.boxplot(x=df_train[col])
    plt.title(col)
plt.tight_layout()
plt.show()


## Outlier Removal-Processing Fees:

In [ ]:
df_train.processing_fee.describe()

In [ ]:
df_train[(df_train.processing_fee/df_train.loan_amount)>0.03][["loan_amount","processing_fee"]]

In [ ]:
df_train_1=df_train[df_train.processing_fee/df_train.loan_amount<0.03].copy()
df_train_1.shape

In [ ]:
df_test=df_test[df_test.processing_fee/df_test.loan_amount<0.03].copy()
df_test.shape

# CHECKING DATA VALIDATION RULES:

In [ ]:
#Rule1:GST should not be more than 20%:
df_train_1[(df_train_1.gst/df_train_1.loan_amount)>0.2].shape

In [ ]:
#Rule 2:Net disbursement should not be higher than loan_amount:
df_train_1[df_train_1.net_disbursement>df_train_1.loan_amount].shape

In [ ]:
columns_categorical

In [ ]:
for col in columns_categorical:
    print(col,"------------->",df_train_1[col].unique())

In [ ]:
#Fixing errors in Loan Purpose Column:
df_train_1['loan_purpose']=df_train_1['loan_purpose'].replace('Personaal','Personal')
df_train_1['loan_purpose'].unique()

In [ ]:
df_test['loan_purpose']=df_test['loan_purpose'].replace('Personaal','Personal')
df_test['loan_purpose'].unique()

# EXPLORATORY DATA ANALYSIS (EDA):

In [ ]:
columns_continuous

In [ ]:
# Exploring the age column:
df_train_1.groupby("default")['age'].describe()

In [ ]:
plt.figure(figsize=(5,4))
sns.kdeplot(df_train_1['age'][df_train_1['default']==0],fill=True,label='default=0')
sns.kdeplot(df_train_1['age'][df_train_1['default']==1],fill=True,label='default=1')
plt.title(f"Age KDE Plot with Hue by default")
plt.legend()
plt.show()

In [ ]:
# Kde for all the columns:
plt.figure(figsize=(20,20))# Width,height in inches
for i,col in enumerate(columns_continuous):
    plt.subplot(6,4,i+1)#1 row,4columns,ithsubplot
    sns.kdeplot(df_train_1[col][df_train_1['default']==0],fill=True,label='default=0')
    sns.kdeplot(df_train_1[col][df_train_1['default']==1],fill=True,label='default=1')
    plt.title(col)        
    plt.xlabel('')
    
plt.tight_layout()
plt.show()

# FEATURE ENGINEERING:

In [ ]:
df_train_1[["loan_amount", "income"]].head(3)

In [ ]:
df_train_1['loan_to_income']=round(df_train_1['loan_amount']/df_train_1['income'],2)
df_train_1['loan_to_income'].describe()

In [ ]:
df_test['loan_to_income']=round(df_test['loan_amount']/df_test['income'],2)

In [ ]:
plt.figure(figsize=(8,4))
sns.kdeplot(df_train_1['loan_to_income'][df_train_1['default'] == 0],fill=True,label='default=0')
sns.kdeplot(df_train_1['loan_to_income'][df_train_1['default'] == 1],fill=True,label='default=1')
plt.title(f"Loan to Income Ratio (LTI) KDE Plot with Hue by default")
plt.legend()
plt.show()

In [ ]:
# Generating Delinquency Ratio:
df_train_1['delinquency_ratio']=(df_train_1['delinquent_months']*100/df_train_1['total_loan_months']).round(1)
df_test['delinquency_ratio']=(df_test['delinquent_months']*100/df_test['total_loan_months']).round(1)

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(df_train_1['delinquency_ratio'][df_train_1['default']==0],fill=True,label='default=0')
sns.kdeplot(df_train_1['delinquency_ratio'][df_train_1['default']==1],fill=True,label='default=1')
plt.title(f"Delinquency Ratio KDE Plot with Hue by default")
plt.legend()
plt.show()

In [ ]:
# Generating Avg Dpd Per Delinquency:
df_train_1['avg_dpd_per_delinquency']=np.where(
    df_train_1['delinquent_months']!=0,(df_train_1['total_dpd']/df_train_1['delinquent_months']).round(1)
    ,0
)

In [ ]:
df_test['avg_dpd_per_delinquency']=np.where(
    df_test['delinquent_months']!=0,(df_test['total_dpd']/df_test['delinquent_months']).round(1)
    ,0
)

In [ ]:
plt.figure(figsize=(8,4))
sns.kdeplot(df_train_1['avg_dpd_per_delinquency'][df_train_1['default']==0],fill=True,label='default=0')
sns.kdeplot(df_train_1['avg_dpd_per_delinquency'][df_train_1['default']==1],fill=True,label='default=1')
plt.title(f"Avg DPD Per Delinquency Ratio KDE Plot with Hue by default")
plt.legend()
plt.show()

In [ ]:
# Removing the columns that are just unique ids and dont have influence on target:
df_train_1.columns

In [ ]:
df_train_2=df_train_1.drop(['cust_id','loan_id'],axis="columns")

In [ ]:
df_test=df_test.drop(['cust_id','loan_id'],axis="columns")

In [ ]:
# Removing Columns that are not required:

df_train_3=df_train_2.drop(['disbursal_date','installment_start_dt','loan_amount','income', 
                              'total_loan_months','delinquent_months','total_dpd'],axis="columns")

In [ ]:
df_test=df_test.drop(['disbursal_date','installment_start_dt','loan_amount','income', 
                              'total_loan_months','delinquent_months','total_dpd'],axis="columns")

In [ ]:
df_train_3.columns

In [ ]:
df_train_3.select_dtypes(['int64','float64']).columns

In [ ]:
# VIF to measure multi co-linearity:

X_train=df_train_3.drop('default',axis='columns')
y_train=df_train_3['default']

In [ ]:
from sklearn.preprocessing import MinMaxScaler
cols_to_scale=X_train.select_dtypes(['int64','float64']).columns
scaler=MinMaxScaler()
X_train[cols_to_scale]=scaler.fit_transform(X_train[cols_to_scale])

In [ ]:
X_train.describe()

In [ ]:
X_test=df_test.drop('default',axis='columns')
y_test=df_test['default']

X_test[cols_to_scale]=scaler.transform(X_test[cols_to_scale])
X_test.describe()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
def calculate_vif(data):
    vif_df=pd.DataFrame()
    vif_df['Column']=data.columns
    vif_df['VIF']=[variance_inflation_factor(data.values,i) for i in range(data.shape[1])]
    return vif_df

In [ ]:
calculate_vif(X_train[cols_to_scale])

In [ ]:
features_to_drop_vif=['sanction_amount','processing_fee','gst','net_disbursement','principal_outstanding']
X_train_1=X_train.drop(features_to_drop_vif,axis='columns')

In [ ]:
numeric_columns=X_train_1.select_dtypes(['int64','float64']).columns
numeric_columns

In [ ]:
vif_df=calculate_vif(X_train_1[numeric_columns])
vif_df

In [ ]:
selected_numeric_features_vif=vif_df.Column.values
selected_numeric_features_vif

In [ ]:
numeric_columns

# FEATURE SELECTION:

In [ ]:
plt.figure(figsize=(12,12))
cm=df_train_3[numeric_columns.append(pd.Index(['default']))].corr()
sns.heatmap(cm,annot=True,fmt='0.2f')
plt.xticks(rotation=45,ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
X_train_1.head()

In [ ]:
# Calculate woe and iv:
def calculate_woe_iv(df,feature,target):
    grouped=df.groupby(feature)[target].agg(['count','sum'])
    grouped=grouped.rename(columns={'count':'total','sum':'good'})
    grouped['bad']=grouped['total']-grouped['good']
    
    total_good=grouped['good'].sum()
    total_bad=grouped['bad'].sum()
    
    grouped['good_pct']=grouped['good']/total_good
    grouped['bad_pct']=grouped['bad']/total_bad
    grouped['woe']=np.log(grouped['good_pct']/grouped['bad_pct'])
    grouped['iv']=(grouped['good_pct']-grouped['bad_pct'])*grouped['woe']
    
    grouped['woe']=grouped['woe'].replace([np.inf,-np.inf],0)
    grouped['iv']=grouped['iv'].replace([np.inf,-np.inf],0)
    
    total_iv=grouped['iv'].sum()
    
    return grouped,total_iv

grouped,total_iv=calculate_woe_iv(pd.concat([X_train_1,y_train],axis=1),'loan_purpose','default')
grouped

In [ ]:
X_train_1.info()

In [ ]:
iv_values = {}
for feature in X_train_1.columns:
    if X_train_1[feature].dtype == 'object':
        _, iv = calculate_woe_iv(pd.concat([X_train_1, y_train],axis=1), feature, 'default' )
    else:
        X_binned = pd.cut(X_train_1[feature], bins=10, labels=False)
        _, iv = calculate_woe_iv(pd.concat([X_binned, y_train],axis=1), feature, 'default' )
    iv_values[feature] = iv
iv_values

In [ ]:
pd.set_option('display.float_format',lambda x: '{:.3f}'.format(x))
iv_df=pd.DataFrame(list(iv_values.items()), columns=['Feature', 'IV'])
iv_df=iv_df.sort_values(by='IV', ascending=False)
iv_df

In [ ]:
# selecting features that has IV > 0.02
selected_features_iv=[feature for feature,iv in iv_values.items() if iv > 0.02]
selected_features_iv

# FEATURE ENCODING:

In [ ]:
X_train_reduced=X_train_1[selected_features_iv]
X_test_reduced=X_test[selected_features_iv]

In [ ]:
X_train_encoded=pd.get_dummies(X_train_reduced,drop_first=True)
X_train_encoded.head(3)

In [ ]:
X_test_encoded=pd.get_dummies(X_test_reduced,drop_first=True)
X_test_encoded.head(3)

# MODEL TRAINING:

## 1.Logistic Regression:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model_lr=LogisticRegression()
model_lr.fit(X_train_encoded,y_train)

y_pred=model_lr.predict(X_test_encoded)
report=classification_report(y_test,y_pred)
print(report)

In [ ]:
feature_importance=model_lr.coef_[0]
# Create a DataFrame for easier handling
coef_df=pd.DataFrame(feature_importance,index=X_train_encoded.columns,columns=['Coefficients'])

# Sort the coefficients for better visualization
coef_df=coef_df.sort_values(by='Coefficients',ascending=True)

# Plotting
plt.figure(figsize=(8,4))
plt.barh(coef_df.index, coef_df['Coefficients'],color='steelblue')
plt.xlabel('Coefficient Value')
plt.title('Feature Importance in Logistic Regression')
plt.show()

In [ ]:
print(coef_df)

## Random Forest:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf=RandomForestClassifier()
model_rf.fit(X_train_encoded,y_train)

y_pred=model_rf.predict(X_test_encoded)
report=classification_report(y_test,y_pred)
print(report)

# Xgboost:

In [ ]:
from xgboost import XGBClassifier

model_xgb= XGBClassifier()
model_xgb.fit(X_train_encoded,y_train)

y_pred=model_xgb.predict(X_test_encoded)
report=classification_report(y_test,y_pred)
print(report)

## Model Evaluation:ROC/AUC

In [ ]:
y_pred=model_lr.predict(X_test_encoded)
report=classification_report(y_test,y_pred)
print(report)

In [ ]:
from sklearn.metrics import roc_curve

probabilities=model_lr.predict_proba(X_test_encoded)[:,1]
fpr,tpr,thresholds=roc_curve(y_test,probabilities)

fpr[:5],tpr[:5],thresholds[:5]

In [ ]:
from sklearn.metrics import auc

area=auc(fpr,tpr)
area

In [ ]:
plt.figure()
plt.plot(fpr,tpr,color='darkorange',lw=2,label='ROC curve (area = %0.2f)' % area)
plt.plot([0, 1],[0, 1],color='navy',lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

## Saving the model:

In [ ]:
X_test_encoded.head(2)

In [ ]:
X_test_encoded.columns

In [ ]:
cols_to_scale

In [ ]:
X_train_encoded.columns

In [ ]:
from joblib import dump

model_data = {
    'model':model_lr,
    'features':X_train_encoded.columns,
    'scaler':scaler,
    'cols_to_scale':cols_to_scale
}
dump(model_data,'artifacts/model_data.joblib')